In [ ]:
# ! pip install agent-framework-azure-ai -U

In [4]:
import os

from azure.identity.aio import AzureCliCredential
from dotenv import load_dotenv

from agent_framework import AgentRunResponse,ChatAgent,HostedFileSearchTool,HostedVectorStoreContent
from agent_framework.azure import AzureAIAgentClient

ImportError: cannot import name 'AgentRunResponse' from 'agent_framework' (/home/dsa/workspace/ai-agents-for-beginners/venv/lib/python3.12/site-packages/agent_framework/__init__.py)

In [3]:
load_dotenv()

True

In [5]:
async def create_vector_store(client: AzureAIAgentClient) -> tuple[str, HostedVectorStoreContent]:
    """创建一个包含示例文档的向量存储。"""
    file_path = './document.md'
    file = await client.project_client.agents.files.upload_and_poll(file_path=file_path, purpose="assistants")
    print(f"上传文件，文件ID: {file.id}")


    vector_store = await client.project_client.agents.vector_stores.create_and_poll(file_ids=[file.id], name="graph_knowledge_base")

    print(f"创建向量存储，ID: {vector_store.id}")


    return file.id, HostedVectorStoreContent(vector_store_id=vector_store.id)

NameError: name 'AzureAIAgentClient' is not defined

In [5]:
async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as chat_client,
    ):
        file_id, vector_store = await create_vector_store(chat_client)

        file_search = HostedFileSearchTool(inputs=vector_store)
        
        agent = chat_client.create_agent(
            name="PythonRAGDemo",
            instructions="""
                您是一个AI助手，设计用于仅使用从提供的文档中检索的信息回答用户问题。

                - 如果用户的问题无法使用检索到的上下文回答，**您必须明确回答**：
                "抱歉，上传的文档不包含回答该问题所需的信息。"
                - 不要根据一般知识或推理回答。不要做出假设或生成假设性解释。
                - 不要提供未在上传文件内容中明确提及的定义、教程或评论。
                - 如果用户问类似"什么是神经网络？"的问题，而这在上传的文档中没有讨论，请按照上述说明回答。
                - 对于在文档中有相关内容的问题（例如Contoso的旅行保险），请准确回答，并明确引用文档。

                您的行为必须表现得好像除了从上传文档中检索到的信息外，没有其他外部知识。
                """,
            tools=[file_search],  # 代理可用的工具
            tool_choice = "auto",  # 让代理决定何时使用工具
        )
                

        print("代理已创建。现在您可以询问有关上传文档的问题。")

        query = "Can you explain Contoso's travel insurance coverage?"
        async for chunk in agent.run_stream(query, tool_resources={"file_search": {"vector_store_ids": [vector_store.vector_store_id]}}):
                
            if chunk.text:
                print(chunk.text, end="", flush=True)

上传文件，文件ID: assistant-Q8ASMZcgB6AcpZzEfmnyv8
创建向量存储，ID: vs_jiTwe3EBs2xvrbDh18Bt21pB
代理已创建。现在您可以询问有关上传文档的问题。
Contoso的旅行保险包括医疗紧急情况、行程取消和行李丢失的保障。此保障是其高级旅行服务的一部分，还包括个性化行程规划和24/7礼宾支持。Contoso Travel将此保险作为其全球异国情调目的地豪华度假套餐的一部分提供，为旅行者在旅途中提供安心【4:0†document.md】。